In [ ]:
%reset -f

In [ ]:
from pynq import PL
from pynq import (allocate, Overlay)
import numpy as np
from PIL import Image

PL.reset()

In [ ]:
ol = Overlay('resizer-zcu102.bit')

In [ ]:
help(ol)

In [ ]:
img2axis = ol.img2axis_0

In [ ]:
# help(img2axis.register_map)

In [ ]:
def img_to_axis(ip,buffer, eos,frame_cnt):
    
# Configure registers:
    # Write physical address of buffer to data_port register
    ip.register_map.data_port = buffer.physical_address

    # Set end_of_stream 
    ip.register_map.end_of_stream = eos

    # Set frame_no to 88
    ip.register_map.frame_cnt = frame_cnt
    # Start the IP core by setting the ap_start bit in CTRL register
    ip.register_map.CTRL.AP_START=1

In [ ]:


def image_to_RGB(image_fname):
    # === LOAD AND CONVERT IMAGE TO RGB ===
    img = Image.open(f"{image_fname}").convert('RGB') 
    img_np = np.array(img)  # Shape: (H, W, 3), dtype=uint8

    # === PACK RGB TO INT32 ===
    # Format: 0x00RRGGBB (most significant byte can be 0)
    r = img_np[:, :, 0].astype(np.uint32)
    g = img_np[:, :, 1].astype(np.uint32)
    b = img_np[:, :, 2].astype(np.uint32)
    rgb_packed = (b << 16) | (g << 8) | r  # Shape: (H, W)

    # Allocate contiguous buffer with dtype uint32
    buffer = allocate(shape=rgb_packed.shape, dtype=np.uint32)

    # Copy packed pixels into buffer
    np.copyto(buffer, rgb_packed)

    print(f"Packed buffer shape: {buffer.shape}, dtype: {buffer.dtype}")
    return buffer

In [ ]:
from pynq.lib.video import *
vdma = ol.axi_vdma_0

vdma.readchannel.reset()
vdma.readchannel.mode = VideoMode(width=1920, height=1080, bits_per_pixel=24)

vdma.readchannel.start()



In [ ]:
print(f"VDMA.running={vdma.readchannel.running},\nVDMA.activeframe={vdma.readchannel.activeframe},\nVDMA.mode={vdma.readchannel.mode}")

In [ ]:
img_fname='1920x1080-full-hd-nature-landscape.jpg'

In [ ]:
buff_o=image_to_RGB(img_fname)

In [ ]:
img_to_axis(ol.img2axis_0,buff_o,True,4)

In [ ]:
frame = vdma.readchannel.readframe()

In [ ]:
print(f"type(frame)={type(frame)},frame.shape={frame.shape},frame.dtype={frame.dtype}")

In [ ]:
from datetime import datetime

timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

#Convert to NumPy array
frame_np = np.array(frame)
#save image
img = Image.fromarray(frame_np, 'RGB')
#img.save(f"{timestamp}-ouput.png")
img.save(f"image-output.png")

# playground

In [ ]:
hasattr(ol.axi_vdma_0, 'write')  # should return True


In [ ]:
img_to_axis(ol.img2axis_0,buff_o,True,88)

In [ ]:
help(VideoMode)

In [ ]:
dir(ol.axi_vdma_0)

In [ ]:
ol.axi_vdma_0.framecount

In [ ]:
len(ol.axi_vdma_0.readchannel._frames)